## Python Analytics Practice -- NumPy, Pandas, Matplotlib, Seaboarn + EDA & Post-Campaign Analysis

### Domain: Ecommerce - Fashion/Home/Electronics Retail

**What is it in this notebook**
- Every section has **Q1, Q2, Q3 ..** style questions in Markdown, followed by a code cell with the answer.
- All data is **synthetically generated inside this notebook** (Section 0) so it's fully self-contained - no external files needed, safe to publish on GitHub.
- Domain logic (BNPL = buy-now-pay-later, categories = Fashion/Home/Electricals/Beauty, channels = Email/Paid Social/Display/Affiliate/TV) is loosely modelled.

**Sections**

0. Synthetic Data Generation (Customers, Products, Campaigns, Exposures, Orders)
1. Numpy Fundamentals (10 Q&A)
2. Pandas Data Wrangling (15 Q&A)
3. Matplotlib Visualization (10 Q&A)
4. Seaborn Visualization (10 Q&A)
5. In-Depth EDA - "Current Market" Style Analysis (10&A)
6. Campaign / Post-Campaign Analysis - the main focus (15 Q&A)
7. Bonus: Mini Take-Home Style Challenge

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 140)
sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['figure.figsize'] = (10,15)

RNG = np.random.default_rng(42)
print("Environment ready.")

Environment ready.


## Section 0 - Syntthetic Data Generation

We build 5 linked tables, similar to a star schema.

| Table | Grain | Key fields |
|---|---|---|
| `customers` | 1 row per customer | region, age_band, is_bnpl_user, acquisition_channel |
| `products` | 1 row per SKU | category, sub_category, brand, price, margin_pct |
| `campaigns` | 1 row per campaign | channel, objective, budget, start/end date |
| `exposures` | 1 row per customer-campaign touch | impressions, clicks, exposure_date |
| `orders` | 1 row per order line | revenue, qty, campaign attribution, returned_flag |

The date range covers **12 months** with realistic seasonality spikes around **Black Friday (Nov)** and **Christmas (Dec)**, since that's when a retailer like "`X Group`" sees its biggest campaign activity.

In [2]:
# ----- 0.1 Customers -----
N_CUSTOMERS = 8000

regions = ['London', 'South East', 'North West', 'Midlands', 'Scotland', 'Wales', 'South West', 'North East']
region_weights = [0.18, 0.15, 0.14, 0.16, 0.10, 0.07, 0.10, 0.10]

age_bands = ['18-24', '25-34', '35-44', '45-54', '55-64', '65+']
age_weights = [0.10, 0.24, 0.23, 0.19, 0.14, 0.10]

acq_channels = ['Paid Social', 'Email', 'Affiliate', 'Organic Search', 'Display', 'TV/Brand']
acq_weights = [0.28, 0.15, 0.12, 0.20, 0.10, 0.15]

signup_start = datetime(2023, 1, 1)
signup_days = RNG.integers(0, 105, N_CUSTOMERS) # up to 3 yrs of tenure

customers = pd.DataFrame({
    'customer_id': [f'CUST{100000+i}' for i in range(N_CUSTOMERS)],
    'signup_date': [signup_start + timedelta(days=int(d)) for d in signup_days],
    'region': RNG.choice(regions, N_CUSTOMERS, p=region_weights),
    'age_band': RNG.choice(age_bands, N_CUSTOMERS, p=age_weights),
    'acquisition_channel': RNG.choice(acq_channels, N_CUSTOMERS, p=acq_weights),
    'is_bnpl_user': RNG.choice([1, 0], N_CUSTOMERS, p=[0.62, 0.38]) # here bnpl penetration is high
})

customers['tenure_days'] = (datetime(2025,12,31) - customers['signup_date']).dt.days

print(customers.shape)
customers.head()

(8000, 7)


,customer_id,signup_date,region,age_band,acquisition_channel,is_bnpl_user,tenure_days
0,CUST100000,2023-01-10,Scotland,35-44,Organic Search,0,1086
1,CUST100001,2023-03-23,North West,35-44,TV/Brand,0,1014
2,CUST100002,2023-03-10,South West,18-24,Paid Social,0,1027
3,CUST100003,2023-02-16,South East,65+,Paid Social,1,1049
4,CUST100004,2023-02-15,Midlands,25-34,Organic Search,0,1050


In [3]:
# ---- 0.2 Products ----
category_map = {
    'Fashion': ['Womenswear', 'Menswear', 'Kidswear', 'Footwear'],
    'Home': ['Furniture', 'Bedding', 'Kitchenware', 'Decor'],
    'Electricals': ['TV & Audio', 'Small Appliances', 'Large Appliances', 'Tech Accessories'],
    'Beauty': ['Skincare', 'Haircare', 'Fragrance']
}

rows = []
pid = 0
for cat, subs in category_map.items():
    for sub in subs:
        n_skus = RNG.integers(15, 30)
        for _ in range(n_skus):
            base_price = {
                'Fashion': RNG.uniform(12, 120),
                'Home': RNG.uniform(15, 600),
                'Electricals': RNG.uniform(20, 1200),
                'Beauty': RNG.uniform(5,80),
            }[cat]
            rows.append({
                'product_id': f'SKU{pid:05d}',
                'category': cat,
                'sub_category': sub,
                'brand': RNG.choice(['House Brand', 'Premium Partner', 'Budget Line', 'Licensed Brand']),
                'price': round(base_price, 2),
                'margin_pct': round(RNG.uniform(0.18, 0.55), 3),
            })
            pid += 1

products = pd.DataFrame(rows)
print(products.shape)
products.head()

(328, 6)


,product_id,category,sub_category,brand,price,margin_pct
0,SKU00000,Fashion,Womenswear,Licensed Brand,30.44,0.486
1,SKU00001,Fashion,Womenswear,Premium Partner,108.42,0.354
2,SKU00002,Fashion,Womenswear,Budget Line,63.34,0.201
3,SKU00003,Fashion,Womenswear,House Brand,73.58,0.359
4,SKU00004,Fashion,Womenswear,Budget Line,22.89,0.377


In [4]:
# ---- 0.3 Campaigns ----
campaigns_defs = [
    ('CMP001',  'Spring Refresh - Home',            'Email',        'Conversion', '2025-03-03', '2025-03-17', 18000),
    ('CMP002',  'Easter Fashion Push',              'Paid Social',  'Conversion', '2025-04-07', '2025-04-21', 42000),
    ('CMP003',  'Summer Electricals Sale',          'Display',      'Conversion', '2025-06-02', '2025-06-16', 35000),
    ('CMP004',  'Back to School',                   'Paid Social',  'Conversion', '2025-08-18', '2025-09-01', 30000),
    ('CMP005',  'Very Pay Awarness',                'TV',           'Awarness',   '2025-09-15', '2025-10-05', 90000),
    ('CMP006',  'Black Friday Blitzz',              'Paid Social',  'Conversion', '2025-11-21', '2025-11-29', 120000),
    ('CMP007',  'Black Friday Email Blast',         'Email',        'Conversion', '2025-11-24', '2025-11-29', 8000),
    ('CMP008',  'Christmas Gift Guide',             'Display',      'Conversion', '2025-12-01', '2025-12-20', 65000),
    ('CMP009',  'Boxing Day Clearance',             'Email',        'Conversion', '2025-12-26', '2025-12-31', 12000),
    ('CMP010',  'New Year New You - Beauty',        'Affiliate',     'Conversion', '2026-01-01', '2026-01-19', 20000),
]
campaigns = pd.DataFrame(campaigns_defs, columns=[
    'campaign_id', 'campaign_name', 'channel', 'objective', 'start_date', 'end_date', 'budget_gbp'
])
campaigns['start_date'] = pd.to_datetime(campaigns['start_date'])
campaigns['end_date'] = pd.to_datetime(campaigns['end_date'])
campaigns['duration_days'] = (campaigns['end_date'] - campaigns['start_date']).dt.days + 1
campaigns

,campaign_id,campaign_name,channel,objective,start_date,end_date,budget_gbp,duration_days
0,CMP001,Spring Refresh - Home,Email,Conversion,2025-03-03,2025-03-17,18000,15
1,CMP002,Easter Fashion Push,Paid Social,Conversion,2025-04-07,2025-04-21,42000,15
2,CMP003,Summer Electricals Sale,Display,Conversion,2025-06-02,2025-06-16,35000,15
3,CMP004,Back to School,Paid Social,Conversion,2025-08-18,2025-09-01,30000,15
4,CMP005,Very Pay Awarness,TV,Awarness,2025-09-15,2025-10-05,90000,21
5,CMP006,Black Friday Blitzz,Paid Social,Conversion,2025-11-21,2025-11-29,120000,9
6,CMP007,Black Friday Email Blast,Email,Conversion,2025-11-24,2025-11-29,8000,6
7,CMP008,Christmas Gift Guide,Display,Conversion,2025-12-01,2025-12-20,65000,20
8,CMP009,Boxing Day Clearance,Email,Conversion,2025-12-26,2025-12-31,12000,6
9,CMP010,New Year New You - Beauty,Affiliate,Conversion,2026-01-01,2026-01-19,20000,19


In [5]:
# ---- 0.4 Campaign Exposure (impressions/clicks per customer per campaign) ----
exposure_rows = []
for _, c in campaigns.iterrows():
    # not every customer is exposed - sample a target audience size depending on channel reach
    reach_frac = {'TV': 0.55, 'Display': 0.40, 'Paid Social': 0.35, 'Email': 0.30, 'Affiliate': 0.15} [c['channel']]
    audience = customers.sample(frac=reach_frac, random_state=RNG.integers(0, 99999))['customer_id'].values
    n = len(audience)
    exp_dates = c['start_date'] + pd.to_timedelta(RNG.integers(0, c['duration_days'], n), unit='D')
    impressions = RNG.poisson(lam={'TV': 6, 'Display': 8, 'Paid Social': 5, 'Email': 2, 'Affiliate': 3}[c['channel']], size=n)+1
    # base CTR varies by channel, with noise
    base_ctr = {'TV': 0.02, 'Display': 0.012, 'Paid Social': 0.028, 'Email': 0.045, 'Affiliate': 0.020}[c['channel']]
    clicks = RNG.binomial(impressions, np.clip(base_ctr + RNG.normal(0, 0.005, n), 0.0005, None))
    exposure_rows.append(pd.DataFrame({
        'customer_id': audience,
        'campaign_id': c['campaign_id'],
        'exposure_date': exp_dates,
        'impressions': impressions,
        'clicks': clicks
    }))

exposures = pd.concat(exposure_rows, ignore_index=True)
print(exposures.shape)
exposures.head()

(27600, 5)


,customer_id,campaign_id,exposure_date,impressions,clicks
0,CUST100624,CMP001,2025-03-14,2,0
1,CUST103083,CMP001,2025-03-12,1,1
2,CUST105840,CMP001,2025-03-05,1,0
3,CUST102623,CMP001,2025-03-12,2,0
4,CUST106888,CMP001,2025-03-15,4,0


In [6]:
# ---- 0.5 Orders (12 months, with seasonality + campaign-attributed uplift) ----
date_range = pd.date_range('2025-01-01', '2026-01-31', freq='D')

# baseline daily demand multiplier (weekday effect + monthly seasonality incl. Black Friday/Xmas spike)
def seasonality(d):
    month_factor = {1: 1.1, 2: 0.85, 3: 0.95, 4: 1.0, 5: 0.95, 6: 1.0, 7: 0.9, 8: 1.0,
                    9: 1.0, 10: 1.05, 11: 1.6, 12: 1.9}[d.month]
    bf_boost = 2.4 if (d.month == 11 and 21 <= d.day <= 29) else 1.0
    xmas_boost = 1.6 if (d.month == 12 and 1 <= d.day <= 20) else 1.0
    weekend_boost = 1.15 if d.weekday() >= 5 else 1.0
    return month_factor * bf_boost * xmas_boost * weekend_boost

order_rows = []
oid = 0
clicked_pairs = exposures[exposures['clicks']>0][['customer_id', 'campaign_id', 'exposure_date']]
clicked_set = clicked_pairs.groupby(['customer_id']).apply(lambda g: list(zip(g['campaign_id'], g['exposure_date'])), include_groups=False).to_dict()

cust_ids = customers['customer_id'].values
prod_sample = products.sample(2500, replace=True, random_state=1).reset_index(drop=True)

for d in date_range:
    n_orders_today = int(RNG.poisson(70 * seasonality(d)))
    sampled_custs = RNG.choice(cust_ids, n_orders_today)
    sampled_prods = products.sample(n_orders_today, replace=True, weights=None, random_state=RNG.integers(0,99999))
    for cust, (_, prod) in zip(sampled_custs, sampled_prods.iterrows()):
        qty = RNG.choice([1,1,1,2,2,3], 1)[0]
        # check if this customer cicked a campaign within +/- 5 days of order -> attribute ordder to campaign
        attributed_campaign = None
        if cust in clicked_set:
            for camp_id, exp_date in clicked_set[cust]:
                if abs((d - exp_date).days) <=5:
                    attributed_campaign = camp_id
                    break
        returned = RNG.choice([1,0], p=[0.12, 0.88])
        order_rows.append((
            f'ORD{oid:07d}', cust, d, prod['product_id'], int(qty),
            round(prod['price'] * qty, 2), attributed_campaign,
            RNG.choice(['Very Pay (BNPL)', 'Debit/Credit Card', 'PayPal'], p=[0.55, 0.35, 0.10]),
            returned
        ))
        oid += 1

orders = pd.DataFrame(order_rows, columns=[
    'order_id', 'customer_id', 'order_date', 'product_id', 'qty', 'revenue',
    'campaign_id', 'payment_method', 'returned_flag'
])
print(orders.shape)
orders.head()

(35300, 9)


,order_id,customer_id,order_date,product_id,qty,revenue,campaign_id,payment_method,returned_flag
0,ORD0000000,CUST103655,2025-01-01,SKU00304,1,51.26,None,Very Pay (BNPL),0
1,ORD0000001,CUST105549,2025-01-01,SKU00013,1,107.08,None,PayPal,0
2,ORD0000002,CUST101099,2025-01-01,SKU00280,1,20.55,None,PayPal,0
3,ORD0000003,CUST101317,2025-01-01,SKU00239,1,622.45,None,Debit/Credit Card,0
4,ORD0000004,CUST101008,2025-01-01,SKU00076,1,33.28,None,Very Pay (BNPL),0


In [7]:
# Quick sanity checks before we start practicing
print('Date range: ', orders['order_date'].min(), 'to', orders['order_date'].max())
print('Total revenue (GBP): {:,.0f}'.format(orders['revenue'].sum()))
print('Orders attributed to a campaign: {:,} ({:.1%})'.format(
    orders['campaign_id'].notna().sum(), orders['campaign_id'].notna().mean()))
print("Missing value check:")
print(orders.isna().sum())

Date range:  2025-01-01 00:00:00 to 2026-01-31 00:00:00
Total revenue (GBP): 15,735,616
Orders attributed to a campaign: 641 (1.8%)
Missing value check:
order_id              0
customer_id           0
order_date            0
product_id            0
qty                   0
revenue               0
campaign_id       34659
payment_method        0
returned_flag         0
dtype: int64


**Note on realism:** This synthetic data - the actual click-through rates, attribution window, and conversion uplift here are simplified for practice purposes. In a ecommerce / analytics learning, you'd reference your own observed metrics (eg: attribution windows, MMM vs last-touch databases) rather than the exact numbers below. Use this dataset to **practice the mechanics** of the analysis, not to memorize these numbers as facts.

## Section 1 - NumPy Fundamentals (10 Q&A)

We'll work with `orders['revenue']` and related arrays converted to NumPy for these exercises, since raw array operations are still common in performance-sensitive data pipelines and in interview SQL/Python crossover questions.

In [8]:
revenue_arr = orders['revenue'].to_numpy()
qty_arr = orders['qty'].to_numpy()
print(revenue_arr[:5], qty_arr[:5])

[ 51.26 107.08  20.55 622.45  33.28] [1 1 1 1 1]


### **NumPy Practice Questions**

**Q1: What is the mean, median and standard deviation of order revenue?**

In [9]:
mean_rev = np.mean(revenue_arr)
median_rev = np.median(revenue_arr)
std_rev = np.std(revenue_arr)
print(f"Mean: {mean_rev:.2f}, Median: {median_rev:.2f}, Standard Deviation: {std_rev:.2f}")

Mean: 445.77, Median: 201.84, Standard Deviation: 565.19


**Q2: Find the 90th and 99th percentile of order revenue - useful for spotting high-value order thresholds.**

In [10]:
p90, p99 = np.percentile(revenue_arr, [90,99])
print(f"90th percentile: {p90:.2f}, 99th percentile: {p99:.2f}")

90th percentile: 1162.96, 99th percentile: 2565.24


**Q3: Use NumPy boolean indexing to count how many orders have revenue above the 99th percentile(potential outliers/fraud-check candidates)**

In [11]:
threshold = np.percentile(revenue_arr, 99)
n_outliers = np.sum(revenue_arr > threshold)
print(f"Orders above 99th percentile (>{threshold:.2f}): {n_outliers}")

Orders above 99th percentile (>2565.24): 343


**Q4: Compute average order value (AOV) per order using vectorized division (revenue/qty), without using a Python loop.**

In [12]:
aov_per_unit = revenue_arr / qty_arr
print(f"Mean unit price across orders: {np.mean(aov_per_unit):.2f}")

Mean unit price across orders: 268.04


**Q5: Use `np.where` to create a flag array marking orders as 'High Value' (>150) vs 'Standard'**

In [13]:
value_flag = np.where(orders['revenue']>150, 'High Value', 'Standard')
unique, counts = np.unique(value_flag, return_counts=True)
print(dict(zip(unique, counts)))

{np.str_('High Value'): np.int64(19632), np.str_('Standard'): np.int64(15668)}


**Q6: Simulate 10,000 bootstrap samples of mean revenue (sample size 500) to estimate a 95% confidence interval for AOV - a technique useful for post-campaign significance testing later.**

In [14]:
boot_means = np.array([
    RNG.choice(revenue_arr, size=500, replace=True).mean()
    for _ in range(10000)
])
ci_low, ci_high = np.percentile(boot_means, [2.5, 97.5])
print(f"Bootstrap 95% CI for AOV: ({ci_low:.2f}, {ci_high:.2f})")

Bootstrap 95% CI for AOV: (397.26, 497.18)


**Q7. Reshape: take the first 5000 revenue values and reshape into a (500, 10) matrix, then compute the row-wise sum (simulating 'revenue per 10-order batch').**

In [15]:
matrix = revenue_arr[:5000].reshape(500,10)
row_sums = matrix.sum(axis=1)
print(row_sums[:5], row_sums.shape)

[1823.17 3838.44 5436.05 1728.12 2899.7 ] (500,)


**Q8. Use `np.corrcoef` to check the correlation between `qty` and `revenue` across all orders.**

In [16]:
corr = np.corrcoef(qty_arr, revenue_arr)[0, 1]
print(f"Correlation between qty and revenue: {corr:.3f}")

Correlation between qty and revenue: 0.352


**Q9. Use `np.cumsum` to compute a running total of revenue across the first 30 days of January 2025 orders (sorted by date) - similar to a running revenue dashboard tile.**

In [42]:
jan = orders[(orders['order_date'] >= '2025-01-01') & (orders['order_date'] <= '2025-01-31')].sort_values('order_date')
daily_rev = jan.groupby('order_date')['revenue'].sum().to_numpy()
running_total = np.cumsum(daily_rev)
print(running_total[:10])

[ 31897.21  70822.64 103237.19 147443.86 200162.29 220638.   251579.94
 288500.91 322691.46 357587.49]


**Q10. Without pandas, use pure NumPy (`np.unique` with `return_counts`) to count how many orders used each payment method.**

In [56]:
pm_arr = orders['payment_method'].to_numpy()
methods, counts = np.unique(pm_arr, return_counts=True)
for m, c in zip(methods, counts):
    print(f"{m}: {c:,} ({c/len(pm_arr):.1%})")

Debit/Credit Card: 12,352 (35.0%)
PayPal: 3,494 (9.9%)
Very Pay (BNPL): 19,454 (55.1%)


## Section 2 - Pandas Data Wrangling (15 Q&A)

Now we work across the full relational structure: `customers`, `products`, `campaigns`, `exposures`, `orders` - same kind of joins you've been practicing in SQL, now in pandas.

**Q1. Merge `orders` with `products` to get category and margin info, then compute total revenue by category.**

In [79]:
orders_p = orders.merge(products[['product_id','category','sub_category','margin_pct']], how='left', on='product_id')
cat_rev = orders_p.groupby('category', as_index=False)['revenue'].sum().sort_values('revenue',ascending=False)
cat_rev

,category,revenue
1,Electricals,9611850.83
3,Home,4541725.59
2,Fashion,1070802.73
0,Beauty,511236.37


**Q2. Compute estimated gross profit per order (`revenue * margin_pct`) and final total estimated profit by category.**

In [91]:
orders_p = orders.merge(products[['product_id','category','sub_category','margin_pct']], how='left', on='product_id')
orders_p['gross_profit'] = round(orders_p['revenue'] * orders_p['margin_pct'], 2)
orders_p.groupby('category', as_index=False).agg({'gross_profit':'sum'}).sort_values('gross_profit',ascending=False)

,category,gross_profit
1,Electricals,3456226.19
3,Home,1658173.91
2,Fashion,403127.39
0,Beauty,183024.87


**Q3. Merge `orders` with `customers` to find total revenue by region and BNPL usage (is_bnpl_user).**

In [103]:
orders_c = orders.merge(customers[['customer_id','region','is_bnpl_user','age_band']], on='customer_id', how='left')
region_bnpl = orders_c.groupby(['region','is_bnpl_user'], as_index=False)['revenue'].sum()
region_bnpl.pivot(index='region', columns='is_bnpl_user', values='revenue').round(0)

is_bnpl_user,0,1
region,,
London,1033839.0,1828011.0
Midlands,986261.0,1514599.0
North East,539228.0,996054.0
North West,807133.0,1336837.0
Scotland,637430.0,1121011.0
South East,946089.0,1443504.0
South West,540408.0,924500.0
Wales,365094.0,715618.0


**Q4. Use `pd.cut` to bucket customers into tenure cohorts ('0-90d','91-365d','1-2yr','2yr+') and find average revenue per order by cohort.**

In [155]:
bins = [-1, 90, 365, 730, 10000]
labels = ['0-90d', '91-365d', '1-2yr', '2+yr']
customers['tenure_cohort'] = pd.cut(customers['tenure_days'], bins=bins, labels=labels)
orders_c = orders.merge(customers[['customer_id','tenure_cohort']], on='customer_id', how='left')
orders_c.groupby('tenure_cohort', as_index=False, observed=True)['revenue'].mean().round(2)

,tenure_cohort,revenue
0,2+yr,445.77


**Q5. Find the top 10 customers by total lifetime revenue, and what % of total company revenue they represent.**

In [156]:
customers.head(2)

,customer_id,signup_date,region,age_band,acquisition_channel,is_bnpl_user,tenure_days,tenure_cohort
0,CUST100000,2023-01-10,Scotland,35-44,Organic Search,0,1086,2+yr
1,CUST100001,2023-03-23,North West,35-44,TV/Brand,0,1014,2+yr


In [157]:
orders.head(2)

,order_id,customer_id,order_date,product_id,qty,revenue,campaign_id,payment_method,returned_flag
0,ORD0000000,CUST103655,2025-01-01,SKU00304,1,51.26,None,Very Pay (BNPL),0
1,ORD0000001,CUST105549,2025-01-01,SKU00013,1,107.08,None,PayPal,0


In [178]:
top10 = orders.groupby('customer_id')['revenue'].sum().sort_values(ascending=False).head(10)
pct_of_total = top10.sum()/orders['revenue'].sum()
print(f"Top 10 customers = {pct_of_total:.2%} of total revenue")
top10

Top 10 customers = 0.62% of total revenue


customer_id
CUST100067    11206.37
CUST104756    10500.46
CUST105203    10494.62
CUST101084     9735.15
CUST101203     9668.50
CUST100113     9542.21
CUST104836     9473.40
CUST100845     9305.91
CUST106642     9113.33
CUST107839     8971.77
Name: revenue, dtype: float64